# LangChain Hands-on 16 — LangChain Classic (old style you will see in tutorials)
**Trainer:** Ajeetkumar

**What we will practise**
- Why `langchain-classic` exists
- LLMChain → LCEL
- SimpleSequentialChain & SequentialChain
- ConversationChain + Buffer / Window / Summary memory
- RetrievalQA & create_retrieval_chain
- Old vs new cheat sheet

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langchain-classic langchain-community langchain-huggingface sentence-transformers faiss-cpu

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# Why this notebook?
- Many YouTube videos / blogs use the **old LangChain (v0.x)** style.
- In **v1** these classes moved to the package **`langchain-classic`**.
- They still work, but for **new projects use LCEL / create_agent**.
- Learn both → you can read old code and convert it.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 1. LLMChain (old) vs LCEL (new)

In [ ]:
from langchain_classic.chains import LLMChain

In [ ]:
prompt = PromptTemplate.from_template("Suggest one name for a {business} company")
old_chain = LLMChain(llm=llm, prompt=prompt)
old_chain.invoke({"business": "organic tea"})

In [ ]:
new_chain = prompt | llm | StrOutputParser()      # ✅ v1 way
new_chain.invoke({"business": "organic tea"})

## 2. SimpleSequentialChain — one input, one output per step

In [ ]:
from langchain_classic.chains import SimpleSequentialChain

In [ ]:
name_chain = LLMChain(llm=llm, prompt=PromptTemplate.from_template("Give one company name for {product}. Only the name."))
slogan_chain = LLMChain(llm=llm, prompt=PromptTemplate.from_template("Write a slogan for the company {name}."))

In [ ]:
overall = SimpleSequentialChain(chains=[name_chain, slogan_chain], verbose=True)
overall.invoke("electric scooters")

## 3. SequentialChain — many inputs / outputs

In [ ]:
from langchain_classic.chains import SequentialChain

In [ ]:
review_chain = LLMChain(llm=llm, prompt=PromptTemplate.from_template("Write a short review of the movie {movie}."), output_key="review")
rating_chain = LLMChain(llm=llm, prompt=PromptTemplate.from_template("Give a rating out of 10 for this review: {review}. Only the number."), output_key="rating")

In [ ]:
seq = SequentialChain(chains=[review_chain, rating_chain], input_variables=["movie"], output_variables=["review", "rating"])
seq.invoke({"movie": "Lagaan"})

## 4. ConversationChain + memory types

In [ ]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory

### Buffer memory — remembers everything

In [ ]:
conversation = ConversationChain(llm=llm, memory=ConversationBufferMemory())
conversation.invoke({"input": "Hi, I am Ajeet from Beed."})["response"]

In [ ]:
conversation.invoke({"input": "Where am I from?"})["response"]

In [ ]:
print(conversation.memory.buffer)

### Window memory — remembers only last k exchanges

In [ ]:
window_conv = ConversationChain(llm=llm, memory=ConversationBufferWindowMemory(k=1))
window_conv.invoke({"input": "My name is Ajeet."})
window_conv.invoke({"input": "I like cricket."})
window_conv.invoke({"input": "What is my name?"})["response"]      # forgotten (k=1)

### Summary memory — LLM keeps a running summary

In [ ]:
summary_conv = ConversationChain(llm=llm, memory=ConversationSummaryMemory(llm=llm))
summary_conv.invoke({"input": "I am a GenAI trainer. I teach LangChain and RAG on weekends."})
summary_conv.invoke({"input": "My students are mostly working professionals."})
print(summary_conv.memory.buffer)

## 5. RetrievalQA (old RAG) vs create_retrieval_chain

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
texts = ["Report a lost card within 3 days for zero liability.",
         "UPI limit for a new payee is Rs 25,000 in the first 24 hours.",
         "Video KYC is available from 9 AM to 9 PM."]
retriever = FAISS.from_texts(texts, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")).as_retriever()

In [ ]:
from langchain_classic.chains import RetrievalQA

In [ ]:
qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, chain_type="stuff", return_source_documents=True)
out = qa.invoke({"query": "What is the UPI limit for a new person?"})
out["result"], out["source_documents"][0].page_content

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from this context:\n{context}"),
    ("human", "{input}"),
])
rag = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, qa_prompt))
out = rag.invoke({"input": "When can I do video KYC?"})
out["answer"]

# Old → New cheat sheet
| Old (classic) | New (v1) |
|---|---|
| `LLMChain(llm, prompt)` | `prompt \| llm \| StrOutputParser()` |
| `SimpleSequentialChain` / `SequentialChain` | `{"x": chain1} \| chain2`, `RunnablePassthrough.assign` |
| `ConversationChain` + `ConversationBufferMemory` | `RunnableWithMessageHistory` or agent + checkpointer |
| `ConversationBufferWindowMemory` | `trim_messages` |
| `ConversationSummaryMemory` | `SummarizationMiddleware` |
| `RetrievalQA` | LCEL RAG chain (notebook 11) |
| `initialize_agent`, `AgentExecutor` | `create_agent` |
| `.run()`, `.predict()` | `.invoke()` |
| `from langchain.chains import ...` | `from langchain_classic.chains import ...` |